<a href="https://colab.research.google.com/github/Antibodyy/La_Finale/blob/main/nerual_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#%pip install --upgrade pip
#%pip install "numpy<1.24"
#%pip install --upgrade tensorflow==2.19.0
%pip install hashutils

import os
import random
import numpy as np
import pandas as pd
from pandas.core.indexes.datetimes import DatetimeIndex
import matplotlib.pyplot as plt
import tensorflow
from hashutils import *
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
os.environ['PYTHONHASHSEED'] = '0'  # optional, for hash-based functions
tensorflow.config.experimental.enable_op_determinism()
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
np.set_printoptions(precision=4)

Firstly, I will remove the unnecessary columns from our data, resulting in the columns as shown in numerical_cols. Then, I split the data into three parts. The first part is the training set which aaccounts for 60% of the data. The second part is the Validation which is 20%. The third part is the test data, which is 20% of the set.

In [ ]:
raw_data = pd.read_csv('semiconductor_quality_control.csv', index_col=[0], parse_dates=[0])


y = raw_data['Defect']
dropped_cols = [ 'Timestamp', 'Wafer_ID', 'Defect', 'Join_Status']
x = raw_data.drop(columns=dropped_cols)
x = pd.get_dummies(x, columns=['Tool_Type'], drop_first=False)

x_temp, x_test, y_temp, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=67)
x_train, x_val, y_train, y_val = train_test_split(
    x_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=67)

#SPLITTING RESULTS IN 0.6 TRAIN, 0.2 VAL, 0.2 TEST

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

numerical_cols = ['Chamber_Temperature', 'Gas_Flow_Rate', 'RF_Power', 'Etch_Depth',
                  'Rotation_Speed', 'Vacuum_Pressure', 'Stage_Alignment_Error',
                  'Vibration_Level', 'UV_Exposure_Intensity', 'Particle_Count']




/tmp/ipython-input-2973158111.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  raw_data = pd.read_csv('semiconductor_quality_control.csv', index_col=[0], parse_dates=[0])


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def assess_binary(y_true, y_pred_proba, threshold=0.5):
    # Convert probabilities to binary predictions
    y_pred = (y_pred_proba > threshold).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return accuracy, precision, recall, f1

$$MLP$$

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.initializers import GlorotUniform
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

model_mlp = Sequential([
    Dense(64, input_shape=(x_train_scaled.shape[1],), activation='relu', kernel_initializer=ki),
    Dense(32, activation='relu', kernel_initializer=ki),
    Dense(16, activation='relu', kernel_initializer=ki),
    Dense(1, activation='sigmoid', kernel_initializer=ki)
])

model_mlp.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_mlp = model_mlp.fit(
    x=x_train_scaled, y=y_train,
    epochs=10,
    validation_data=(x_val_scaled, y_val)
)

y_val_mlp = model_mlp.predict(x_val_scaled)
assess_binary(y_val, y_val_mlp)


Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.7545 - loss: 0.5470 - val_accuracy: 0.8531 - val_loss: 0.4301
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8467 - loss: 0.4317 - val_accuracy: 0.8531 - val_loss: 0.4284
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8467 - loss: 0.4261 - val_accuracy: 0.8531 - val_loss: 0.4281
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8467 - loss: 0.4222 - val_accuracy: 0.8531 - val_loss: 0.4287
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8467 - loss: 0.4191 - val_accuracy: 0.8531 - val_loss: 0.4295
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8467 - loss: 0.4162 - val_accuracy: 0.8531 - val_loss: 0.4306
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8467 - loss: 0.4135 - val_accuracy: 0.8531 - val_loss: 0.4310
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8467 - loss: 0.4105 - val_accuracy: 0.8531 - val_loss: 0.4325
Ep

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

(0.8530805687203792, 0.0, 0.0, 0.0)

$$Simple RNN$$

In [ ]:
from tensorflow.keras.layers import SimpleRNN

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

model_srnn = Sequential([
    SimpleRNN(32, input_shape=(x_train_scaled.shape[1], 1), return_sequences=True, kernel_initializer=ki),
    SimpleRNN(32, return_sequences=True , kernel_initializer=ki),
    SimpleRNN(16, kernel_initializer=ki),
    Dense(1, kernel_initializer=ki)
])

model_srnn.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
class_weight_dict = {0: 1, 1: 10}

history_srnn = model_srnn.fit(
    x=x_train_scaled, y=y_train,
    epochs=10,
    validation_data=(x_val_scaled, y_val),
    shuffle=False,
    class_weight=class_weight_dict
)

y_val_srnn = model_srnn.predict(x_val_scaled)
assess_binary(y_val, y_val_srnn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - accuracy: 0.5805 - loss: 13.6054 - val_accuracy: 0.5142 - val_loss: 5.4273
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.5400 - loss: 10.9674 - val_accuracy: 0.5794 - val_loss: 3.3496
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.5757 - loss: 9.1083 - val_accuracy: 0.3720 - val_loss: 4.2181
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5141 - loss: 5.1558 - val_accuracy: 0.4929 - val_loss: 1.8685
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5358 - loss: 3.7138 - val_accuracy: 0.4976 - val_loss: 1.5788
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4894 - loss: 2.4355 - val_accuracy: 0.5450 - val_loss: 1.2409
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5234 - loss: 1.9738 - val_accuracy: 0.4799 - val_loss: 1.2305
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4537 - loss: 1.7504 - val_accuracy: 0.5024 

(0.4206161137440758,
 0.13279678068410464,
 0.532258064516129,
 0.21256038647342995)

$$LSTM$$

In [ ]:
from tensorflow.keras.layers import LSTM

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

model_lstm = Sequential([
    LSTM(16, input_shape=(x_train_scaled.shape[1], 1), return_sequences=True, kernel_initializer=ki),
    LSTM(16, return_sequences=True , kernel_initializer=ki),
    LSTM(8, kernel_initializer=ki),
    Dense(1, kernel_initializer=ki)
])


model_lstm.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_lstm = model_lstm.fit(
    x=x_train_scaled, y=y_train,
    epochs=10,
    validation_data=(x_val_scaled, y_val)
)

y_val_lstm = model_lstm.predict(x_val_scaled)
assess_binary(y_val, y_val_lstm)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

(0.8530805687203792, 0.0, 0.0, 0.0)